Imports


In [ ]:
import sqlite3
import pandas as pd
import os

DWH = sqlite3.connect("../../Database/DWH.db")
SDM = sqlite3.connect("../../Database/SDM2.db")

sdm_cursor = SDM.cursor()
dwh_cursor = DWH.cursor()


Laad SDM tabellen


In [ ]:
df_klant = pd.read_sql_query("SELECT * FROM Klant", SDM)
df_fiets = pd.read_sql_query("SELECT * FROM Fiets", SDM)
df_monteur = pd.read_sql_query("SELECT * FROM Monteur", SDM)
df_accessoire = pd.read_sql_query("SELECT * FROM Accessoire", SDM)

df_fact_fiets = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fiets_Verkoop", SDM)
df_fact_onderhoud = pd.read_sql_query("SELECT * FROM Onderhoud_Onderhoud", SDM)
df_fact_accessoire = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Accessoire_Verkoop", SDM)


SCD type 2 

In [ ]:
def etl_dim_klant(df):
    for _, row in df.iterrows():

        klantnr = row["klantnr"]

        dwh_cursor.execute("""
            SELECT naam, woonplaats, geslacht, geboortedatum, leeftijdsgroep
            FROM Dim_Klant
            WHERE klantnr = ? AND is_actueel = 1
        """, (klantnr,))
        
        existing = dwh_cursor.fetchone()

        if existing is None:
            # insert new
            dwh_cursor.execute("""
                INSERT INTO Dim_Klant
                VALUES (NULL, ?, ?, ?, ?, ?, ?, DATE('now'), NULL, 1)
            """, (
                klantnr,
                row["naam"],
                row["woonplaats"],
                row["geslacht"],
                row["geboortedatum"],
                row["leeftijdsgroep"]
            ))

        elif existing != (
            row["naam"],
            row["woonplaats"],
            row["geslacht"],
            row["geboortedatum"],
            row["leeftijdsgroep"]
        ):
            # expire old
            dwh_cursor.execute("""
                UPDATE Dim_Klant
                SET geldig_tot = DATE('now'), is_actueel = 0
                WHERE klantnr = ? AND is_actueel = 1
            """, (klantnr,))

            # insert new
            dwh_cursor.execute("""
                INSERT INTO Dim_Klant
                VALUES (NULL, ?, ?, ?, ?, ?, ?, DATE('now'), NULL, 1)
            """, (
                klantnr,
                row["naam"],
                row["woonplaats"],
                row["geslacht"],
                row["geboortedatum"],
                row["leeftijdsgroep"]
            ))

    DWH.commit()
